In [3]:
# 1. Instalasi (jika perlu)
# pip install pandas scikit-learn langdetect textblob transformers torch

import pandas as pd
from langdetect import detect
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob
from transformers import pipeline

# 2. Load & deteksi bahasa (tangani NaN)
df = pd.read_csv('data_stemm.csv')
def detect_lang_safe(text):
    if not isinstance(text, str) or not text.strip(): return 'unknown'
    try: return detect(text)
    except: return 'unknown'
df['lang'] = df['stemmed'].apply(detect_lang_safe)
df = df[df['lang'].isin(['en','id'])].copy()
df = df[df['stemmed'].str.strip() != '']

# 3. Siapkan sentiment analyzers
# English via TextBlob
def sentiment_en(txt):
    p = TextBlob(txt).sentiment.polarity
    return 'positive' if p > 0.1 else 'negative' if p < -0.1 else 'neutral'

# Indonesian via Roberta classifier
sent_id = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    tokenizer="w11wo/indonesian-roberta-base-sentiment-classifier"
)
def sentiment_id(txt):
    out = sent_id(txt[:512])[0]
    lbl = out['label'].lower()
    if 'neg' in lbl: return 'negative'
    if 'pos' in lbl: return 'positive'
    return 'neutral'

# 4. Siapkan emotion analyzer
# English Emotion Model (HuggingFace)
emotion_en = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=True
)
def emotion_en_top1(text):
    try:
        result = emotion_en(text[:512])[0]
        top = max(result, key=lambda x: x['score'])
        return top['label'].lower()
    except:
        return 'unknown'

# Indonesian emotion dengan kamus sederhana
emo_dict_id = {
    'senang': 'joy', 'gembira': 'joy', 'puas': 'joy',
    'marah': 'anger', 'kesal': 'anger',
    'takut': 'fear', 'cemas': 'fear',
    'sedih': 'sadness', 'kecewa': 'sadness',
    'terkejut': 'surprise', 'kaget': 'surprise'
}
def emotion_id_keyword(text):
    for word in emo_dict_id:
        if word in text:
            return emo_dict_id[word]
    return 'neutral'

# 5. Fungsi utama analisis per bahasa
def analyze_language(df, lang_code, n_topics=5, n_top_words=10):
    sub = df[df['lang'] == lang_code].copy()
    texts = sub['stemmed'].tolist()
    
    # a) Vectorize & LDA Topic Modeling
    vect = CountVectorizer(max_df=0.8, min_df=5)
    dtm  = vect.fit_transform(texts)
    lda  = LatentDirichletAllocation(n_components=n_topics, random_state=42)
    lda.fit(dtm)
    
    # b) Assign dominant topic
    topic_dist = lda.transform(dtm)
    sub['dominant_topic'] = topic_dist.argmax(axis=1)
    
    # c) Sentiment analysis
    if lang_code == 'en':
        sub['sentiment'] = sub['stemmed'].apply(sentiment_en)
    else:
        sub['sentiment'] = sub['stemmed'].apply(sentiment_id)

    # d) Emotion analysis
    if lang_code == 'en':
        sub['emotion'] = sub['stemmed'].apply(emotion_en_top1)
    else:
        sub['emotion'] = sub['stemmed'].apply(emotion_id_keyword)

    # e) Summary Sentiment per Topic
    sent_summary = (sub
                    .groupby(['dominant_topic','sentiment'])
                    .size()
                    .unstack(fill_value=0))
    sent_summary['topic_sentiment'] = sent_summary.idxmax(axis=1)

    # f) Summary Emotion per Topic
    emo_summary = (sub
                   .groupby(['dominant_topic','emotion'])
                   .size()
                   .unstack(fill_value=0))
    
    # g) Cetak top words dan label sentimen dominan
    feat = vect.get_feature_names_out()
    print(f"\n=== Language: {lang_code} ===")
    for t in range(n_topics):
        top_idx = lda.components_[t].argsort()[:-n_top_words-1:-1]
        topw    = [feat[i] for i in top_idx]
        sent_lbl = sent_summary.loc[t,'topic_sentiment'] if t in sent_summary.index else 'N/A'
        print(f"Topic {t+1} ({sent_lbl}): {', '.join(topw)}")

    # h) Cetak distribusi
    print("\nDistribusi Sentimen per Topik:")
    print(sent_summary)
    print("\nDistribusi Emosi per Topik:")
    print(emo_summary)
    
    return sub, sent_summary, emo_summary

# 6. Jalankan untuk English dan Indonesian
sub_en, sent_en, emo_en = analyze_language(df, 'en', n_topics=5, n_top_words=10)
sub_id, sent_id, emo_id = analyze_language(df, 'id', n_topics=5, n_top_words=10)


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

c:\Users\HAJRAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

c:\Users\HAJRAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(



=== Language: en ===
Topic 1 (positive): ad, app, heart, use, learn, practic, get, lesson, im, duolingo
Topic 2 (positive): learn, app, languag, duolingo, use, lesson, good, make, way, even
Topic 3 (positive): lesson, get, app, time, xp, use, pay, everi, dont, cant
Topic 4 (positive): learn, word, languag, app, use, also, like, make, im, sentenc
Topic 5 (positive): learn, app, languag, word, like, duolingo, lesson, time, new, ad

Distribusi Sentimen per Topik:
sentiment       negative  neutral  positive topic_sentiment
dominant_topic                                             
0                     23       78       156        positive
1                     18       64       142        positive
2                     16       56        59        positive
3                     23       61        88        positive
4                     15       57       145        positive

Distribusi Emosi per Topik:
emotion         anger  disgust  fear  joy  neutral  sadness  surprise
dominant_topic 